In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
data = pd.read_excel('portatour-Kunden.xlsx')

In [4]:
data.head()

,Kundenname,Straße,Ort,PLZ,Land
0,138395 F & T Terrix Limitid,"Chichester Street, Unit 14 Chichester busines...",Rochdale,OL16 2AU,GB
1,Addagrip Surface,Bell Lane Industrial Estate,"Uckfield, East Sussex",TN22 1QL,GB
2,AHL Architectural Hardware,Boucher Place,BELFAST,BT12 6TA,GB
3,ALIVA UK,Arlington Business Park,"Theale, Berkshire",RG7 4TY,GB
4,ALIVA UK,"Oak Tree Barn, Unit 6",Warrington,WA4 4BX,GB


In [ ]:
API_KEY= "API_KEY_HERE"

In [14]:
from openai import OpenAI
import time

client = OpenAI(api_key=API_KEY)

def research_company(name, street, city, plz, country):
    """Use GPT-5.2 with web search to research a company."""
    prompt = (
        f"Company: '{name}', Address: {street}, {city}, {plz}, {country}.\n"
        f"Context: building materials / construction industry.\n\n"
        f"Answer VERY briefly — max 1 short bullet point per question, no full sentences:\n"
        f"TAETIGKEIT: Core business (max 5 words)\n"
        f"GROESSE: Size (employees/revenue/locations, max 5 words)\n"
        f"PRODUKTPALETTE: List only product categories (e.g. WDVS, Trockenbau, Putz, Fassade, Dämmstoffe). Just comma-separated keywords.\n\n"
        f"Format exactly as:\n"
        f"TAETIGKEIT: ...\n"
        f"GROESSE: ...\n"
        f"PRODUKTPALETTE: ..."
    )

    try:
        response = client.responses.create(
            model="gpt-5.2",
            tools=[{"type": "web_search"}],
            input=prompt,
        )
        return response.output_text
    except Exception as e:
        print(f"Error for {name}: {e}")
        return "TAETIGKEIT: N/A\nGROESSE: N/A\nPRODUKTPALETTE: N/A"

def parse_response(text):
    """Parse the structured response into 3 fields."""
    result = {"Tätigkeit": "", "Größe": "", "Produktpalette": ""}
    for line in text.split("\n"):
        line = line.strip()
        if line.upper().startswith("TAETIGKEIT:"):
            result["Tätigkeit"] = line.split(":", 1)[1].strip()
        elif line.upper().startswith("GROESSE:"):
            result["Größe"] = line.split(":", 1)[1].strip()
        elif line.upper().startswith("PRODUKTPALETTE:"):
            result["Produktpalette"] = line.split(":", 1)[1].strip()
    return result

In [15]:

# TEST: Run only on the first row
row = data.iloc[0]
print(f"Testing: {row['Kundenname']}")
raw = research_company(row["Kundenname"], row["Straße"], row["Ort"], row["PLZ"], row["Land"])
print("\n--- Raw Response ---")
print(raw)
print("\n--- Parsed ---")
parsed = parse_response(raw)
for k, v in parsed.items():
    print(f"{k}: {v}")

Testing: 138395 F & T Terrix Limitid

--- Raw Response ---
TAETIGKEIT: Fassaden-/Putzsysteme Hersteller/Anbieter  
GROESSE: klein; <50 MA; 1 Standort  
PRODUKTPALETTE: Putz, Render, Farbe, Sprayputz, Sprührender, Brick Slips, EWI/WDVS, Innenputz, Fassadensysteme

--- Parsed ---
Tätigkeit: Fassaden-/Putzsysteme Hersteller/Anbieter
Größe: klein; <50 MA; 1 Standort
Produktpalette: Putz, Render, Farbe, Sprayputz, Sprührender, Brick Slips, EWI/WDVS, Innenputz, Fassadensysteme


In [6]:
data = df_enriched.copy()  # to avoid modifying original

NameError: name 'df_enriched' is not defined

In [17]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def process_row(idx, row):
    """Worker function for parallel execution."""
    raw = research_company(row["Kundenname"], row["Straße"], row["Ort"], row["PLZ"], row["Land"])
    parsed = parse_response(raw)
    return idx, row["Kundenname"], parsed

# Run enrichment in parallel with 5 workers
MAX_WORKERS = 20
results = {}
total = len(data)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(process_row, idx, row): idx
        for idx, row in data.iterrows()
    }
    for future in as_completed(futures):
        idx, name, parsed = future.result()
        results[idx] = parsed
        print(f"[{len(results)}/{total}] {name} ✓")

# Build final dataframe with new columns (in original order)
#df_enriched = data.copy()
#df_enriched["Tätigkeit"] = [results[i]["Tätigkeit"] for i in range(total)]
#df_enriched["Größe"] = [results[i]["Größe"] for i in range(total)]
#df_enriched["Produktpalette"] = [results[i]["Produktpalette"] for i in range(total)]

#print(f"\nDone! {len(df_enriched)} companies enriched.")
#df_enriched

[1/180] Belgrade Insulations ✓
[2/180] Bradfords ✓
[3/180] Belmore Supplies ✓
[4/180] Belgrade Insulations ✓
[5/180] BTS INTERIORS LTD. ✓
[6/180] alsecco (U.K.) Ltd. ✓
[7/180] Addagrip Surface ✓
[8/180] AHL Architectural Hardware ✓
[9/180] Belgrade Insulations (138552) ✓
[10/180] British Gypsum PLC. ✓
[11/180] Amaroc -  gelöscht ✓
[12/180] Bryson Products ✓
[13/180] Bluebuild Energy ✓
[14/180] 138395 F & T Terrix Limitid ✓
[15/180] ALIVA UK ✓
[16/180] Bostik Ltd ✓
[17/180] Benchmark Building ✓
[18/180] Baumit  Ltd ✓
[19/180] Allmat (East Surrey) Ltd. ✓
[20/180] ALIVA UK ✓


: 

: 

In [ ]:
pd.read_excel('portatour-Kunden.xlsx').head()

In [12]:
df_enriched.head()

,Kundenname,Straße,Ort,PLZ,Land,Tätigkeit,Größe,Produktpalette
0,138395 F & T Terrix Limitid,"Chichester Street, Unit 14 Chichester busines...",Rochdale,OL16 2AU,GB,"Fassaden-/Putzsysteme, Baustoffe",klein; <50 MA; 1 Standort,"Render, Putz/Spritzputz, Farben/Lacksysteme, K..."
1,Addagrip Surface,Bell Lane Industrial Estate,"Uckfield, East Sussex",TN22 1QL,GB,Resin surfacing systems manufacturer,"<50 employees, 1 UK site","resin-bound surfacing, resin-bonded surfacing,..."
2,AHL Architectural Hardware,Boucher Place,BELFAST,BT12 6TA,GB,Architectural ironmongery supply,Single site; small business,"Schlösser, Zylinder, Schlüssel/Schließsysteme,..."
3,ALIVA UK,Arlington Business Park,"Theale, Berkshire",RG7 4TY,GB,"Fassadenlösungen, hinterlüftete Systeme","2 Standorte, KMU (<50)","ETICS/EWI, Rainscreen, Porzellan/Feinsteinzeug..."
4,ALIVA UK,"Oak Tree Barn, Unit 6",Warrington,WA4 4BX,GB,"Fassadensysteme, EWI, Rainscreen","HQ Süd, Büro/Showroom Nord","ETICS/EWI, Rainscreen-Cladding, Fassadenpaneel..."


In [13]:
df_enriched.to_excel("kunden_enriched.xlsx", index=False)

In [7]:
df_enriched=pd.read_excel("kunden_enriched.xlsx")

In [4]:
df_enriched.head()

,Kundenname,Straße,Ort,PLZ,Land,Tätigkeit,Größe,Produktpalette
0,138395 F & T Terrix Limitid,"Chichester Street, Unit 14 Chichester busines...",Rochdale,OL16 2AU,GB,"Fassaden-/Putzsysteme, Baustoffe",klein; <50 MA; 1 Standort,"Render, Putz/Spritzputz, Farben/Lacksysteme, K..."
1,Addagrip Surface,Bell Lane Industrial Estate,"Uckfield, East Sussex",TN22 1QL,GB,Resin surfacing systems manufacturer,"<50 employees, 1 UK site","resin-bound surfacing, resin-bonded surfacing,..."
2,AHL Architectural Hardware,Boucher Place,BELFAST,BT12 6TA,GB,Architectural ironmongery supply,Single site; small business,"Schlösser, Zylinder, Schlüssel/Schließsysteme,..."
3,ALIVA UK,Arlington Business Park,"Theale, Berkshire",RG7 4TY,GB,"Fassadenlösungen, hinterlüftete Systeme","2 Standorte, KMU (<50)","ETICS/EWI, Rainscreen, Porzellan/Feinsteinzeug..."
4,ALIVA UK,"Oak Tree Barn, Unit 6",Warrington,WA4 4BX,GB,"Fassadensysteme, EWI, Rainscreen","HQ Süd, Büro/Showroom Nord","ETICS/EWI, Rainscreen-Cladding, Fassadenpaneel..."


In [6]:
df_enriched.columns

Index(['Kundenname', 'Straße', 'Ort', 'PLZ', 'Land', 'Tätigkeit', 'Größe',
       'Produktpalette'],
      dtype='object')

In [10]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI

client = OpenAI(api_key=API_KEY)

def fetch_website(name, street, city, plz, country):
    """Use GPT with web search to find the real company website."""
    prompt = (
        f"Find the official website URL for this company:\n"
        f"Company name: '{name}'\n"
        f"Address: {street}, {plz} {city}, {country}\n"
        f"Industry context: building materials / construction (Baustoffhandel, Baustoffe)\n\n"
        f"Rules:\n"
        f"- Return ONLY the base domain URL (e.g. https://www.example.com)\n"
        f"- It MUST be the real, verified website of this specific company\n"
        f"- Do NOT guess or invent URLs\n"
        f"- Do NOT return social media pages, directory listings, or third-party sites\n"
        f"- If you cannot find a verified website, return exactly: NICHT_GEFUNDEN\n\n"
        f"Answer with ONLY the URL or NICHT_GEFUNDEN, nothing else."
    )

    try:
        response = client.responses.create(
            model="gpt-5.2",
            tools=[{"type": "web_search"}],
            input=prompt,
        )
        url = response.output_text.strip()

        # Validate: must look like a real URL and not be the fallback
        if "NICHT_GEFUNDEN" in url.upper() or "nicht" in url.lower():
            return ""
        if not url.startswith("http"):
            # Sometimes the model returns domain without scheme
            if "." in url and " " not in url and len(url) < 100:
                url = "https://" + url
            else:
                return ""
        # Strip trailing slashes/whitespace and remove any surrounding quotes
        url = url.strip().strip('"').strip("'").rstrip("/")
        return url
    except Exception as e:
        print(f"Error for {name}: {e}")
        return ""


def process_website_row(idx, row):
    """Worker function for parallel website lookup."""
    url = fetch_website(row["Kundenname"], row["Straße"], row["Ort"], row["PLZ"], row["Land"])
    return idx, row["Kundenname"], url


# Run website lookup in parallel
MAX_WORKERS = 20
website_results = {}
total = len(df_enriched)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(process_website_row, idx, row): idx
        for idx, row in df_enriched.iterrows()
    }
    for future in as_completed(futures):
        idx, name, url = future.result()
        website_results[idx] = url
        status = url if url else "(not found)"
        print(f"[{len(website_results)}/{total}] {name} → {status}")

# Add website column
df_enriched["Website"] = [website_results[i] for i in range(total)]

found = df_enriched["Website"].astype(bool).sum()
print(f"\nDone! Websites found: {found}/{total}")
df_enriched[["Kundenname", "Website"]].head(20)

[1/180] Baumit  Ltd → https://baumit.co.uk
[2/180] Belgrade Insulations → https://www.belgradeinsulations.com
[3/180] Bluebuild Energy → https://www.bluebuildenergy.com
[4/180] Belgrade Insulations → https://www.belgradeinsulations.com
[5/180] Bostik Ltd → https://www.bostik.com
[6/180] Addagrip Surface → https://addagrip.co.uk
[7/180] ALIVA UK → https://alivauk.com
[8/180] Bryson Products → https://www.bryson.co.uk
[9/180] Belmore Supplies → https://www.belmoretools.co.uk
[10/180] Bradfords → https://www.bradfords.co.uk
[11/180] Belgrade Insulations (138552) → https://www.belgradeinsulations.com
[12/180] alsecco (U.K.) Ltd. → https://alsecco.co.uk
[13/180] ALIVA UK → https://alivauk.com
[14/180] 138395 F & T Terrix Limitid → https://www.ft-terrix.com
[15/180] Amaroc -  gelöscht → (not found)
[16/180] British Gypsum PLC. → https://www.british-gypsum.com
[17/180] Allmat (East Surrey) Ltd. → https://www.allmat-buildingsupplies.co.uk
[18/180] CCF Ltd. → https://www.ccfltd.co.uk
[19/180] C

,Kundenname,Website
0,138395 F & T Terrix Limitid,https://www.ft-terrix.com
1,Addagrip Surface,https://addagrip.co.uk
2,AHL Architectural Hardware,
3,ALIVA UK,https://alivauk.com
4,ALIVA UK,https://alivauk.com
5,Allmat (East Surrey) Ltd.,https://www.allmat-buildingsupplies.co.uk
6,alsecco (U.K.) Ltd.,https://alsecco.co.uk
7,Amaroc - gelöscht,
8,Baumit Ltd,https://baumit.co.uk
9,Belgrade Insulations,https://www.belgradeinsulations.com


In [11]:
df_enriched.head()

,Kundenname,Straße,Ort,PLZ,Land,Tätigkeit,Größe,Produktpalette,Website
0,138395 F & T Terrix Limitid,"Chichester Street, Unit 14 Chichester busines...",Rochdale,OL16 2AU,GB,"Fassaden-/Putzsysteme, Baustoffe",klein; <50 MA; 1 Standort,"Render, Putz/Spritzputz, Farben/Lacksysteme, K...",https://www.ft-terrix.com
1,Addagrip Surface,Bell Lane Industrial Estate,"Uckfield, East Sussex",TN22 1QL,GB,Resin surfacing systems manufacturer,"<50 employees, 1 UK site","resin-bound surfacing, resin-bonded surfacing,...",https://addagrip.co.uk
2,AHL Architectural Hardware,Boucher Place,BELFAST,BT12 6TA,GB,Architectural ironmongery supply,Single site; small business,"Schlösser, Zylinder, Schlüssel/Schließsysteme,...",
3,ALIVA UK,Arlington Business Park,"Theale, Berkshire",RG7 4TY,GB,"Fassadenlösungen, hinterlüftete Systeme","2 Standorte, KMU (<50)","ETICS/EWI, Rainscreen, Porzellan/Feinsteinzeug...",https://alivauk.com
4,ALIVA UK,"Oak Tree Barn, Unit 6",Warrington,WA4 4BX,GB,"Fassadensysteme, EWI, Rainscreen","HQ Süd, Büro/Showroom Nord","ETICS/EWI, Rainscreen-Cladding, Fassadenpaneel...",https://alivauk.com


In [12]:
df_enriched.to_excel("kunden_enriched_with_websites.xlsx", index=False)